# 04 · olmOCR (AllenAI)

**olmOCR** — пайплайн от AllenAI поверх Qwen2-VL-7B-Instruct, дообученный на ~250K страницах. Чекпоинт: [`allenai/olmOCR-7B-0225-preview`](https://huggingface.co/allenai/olmOCR-7B-0225-preview).

Ноутбук берёт subset, сохранённый в `data/subset.json` ноутбуком `01_setup_and_dataset.ipynb`, и прогоняет на нём модель. Результаты записываются в `results/<model>/predictions.jsonl`.

## 1. Установка

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# === Colab / Kaggle bootstrap =================================================
# В Colab клонируем репозиторий проекта (предполагается, что код выложен
# в GitHub) и переходим в его корень. Для локального запуска просто
# проверьте, что текущая рабочая директория — корень ocr_eval/.
import os, sys, pathlib

# Клонируем только если ещё нет (защита от повторного запуска)
if not pathlib.Path('ocr_eval').exists():
    !git clone https://github.com/AStrateg2509/ocr_eval.git

os.chdir('ocr_eval')
sys.path.insert(0, 'src')
print("CWD =", os.getcwd())


## 2. Системные зависимости (poppler нужен для anchor-текста)

In [ ]:
!apt-get -qq install -y poppler-utils ttf-mscorefonts-installer 2>&1 | tail -1
!pip install -q olmocr

In [ ]:
from src.utils import load_config, JsonlWriter, Timer, cuda_free, gpu_info, already_processed_ids
from src.io_records import PredictionRecord
cfg = load_config('configs/olmocr.yaml')
cfg

## 3. Загрузка модели Qwen2-VL

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL_REPO = cfg['model']['hf_repo']
processor = AutoProcessor.from_pretrained(MODEL_REPO)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_REPO,
    torch_dtype=getattr(torch, cfg['model']['torch_dtype']),
    device_map=cfg['model']['device_map'],
).eval()
print(gpu_info())

## 4. Inference

olmOCR работает по схеме: PDF → рендер страницы + anchor-текст из poppler → многостраничный prompt → Qwen2-VL. Для OmniDocBench у нас уже есть PNG-страницы, поэтому используем упрощённый путь без anchor-текста (в отчёте отметить, что это даёт небольшой проигрыш по сравнению с полным пайплайном).

In [ ]:
import json, pathlib
from src.dataset_loader import GroundTruth

DATA_ROOT = pathlib.Path('data/OmniDocBench')
subset = [GroundTruth(**rec) for rec in json.loads(
    pathlib.Path('data/subset.json').read_text(encoding='utf-8'))]
print(f'subset: {len(subset)} страниц')

In [ ]:
import traceback, base64, io
from pathlib import Path
from PIL import Image

out_path = Path(cfg['output']['results_dir']) / 'predictions.jsonl'
out_path.parent.mkdir(parents=True, exist_ok=True)
done = already_processed_ids(out_path)

MAX_NEW = cfg['inference']['max_new_tokens']
TEMP    = cfg['inference']['temperature']

OLMOCR_PROMPT = (
    'Below is the image of one page of a document. Just return the plain text '
    'representation of this document as if you were reading it naturally. '
    'Convert equations to LaTeX and tables to HTML. Do not hallucinate.'
)

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done: continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists(): continue
        rec = PredictionRecord(page_id=gt.page_id, model='olmocr')
        try:
            img = Image.open(img_path).convert('RGB')
            messages = [{
                'role': 'user',
                'content': [
                    {'type': 'image'},
                    {'type': 'text', 'text': OLMOCR_PROMPT},
                ],
            }]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(text=[text], images=[img], padding=True, return_tensors='pt').to(model.device)
            with Timer('infer') as t, torch.no_grad():
                gen = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=TEMP > 0,
                                     temperature=TEMP if TEMP > 0 else 1.0)
            out = processor.batch_decode(
                gen[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
            rec.full_text = out
            rec.raw_output = out
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f'{type(e).__name__}: {e}'
            traceback.print_exc()
        w.write(rec.to_dict())
print('готово →', out_path)

## 5. Освободить GPU

In [ ]:
del model; cuda_free(); print(gpu_info())